# Graph Beckmann Flow for $\Wass_1$

This notebook generates `fig:w1-graph-transport-flow`.  On a finite graph with edge lengths $\ell_e$, the graph Beckmann formulation is
$$
    \Wass_{1,G}(\alpha,\beta)=\min_m\left\{\sum_{e\in E}\ell_e |m_e|:\ \operatorname{div}_G m=\alpha-\beta\right\}.
$$
The figure uses a Delaunay graph large enough to show routing.  Surplus and deficit are localized on five vertices each, and the optimal signed edge flow is drawn by arrow orientation and width.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from matplotlib.colors import ListedColormap, to_rgb
from matplotlib.patches import FancyArrowPatch
from scipy.optimize import linprog
from scipy.spatial import Delaunay

ROOT = Path.cwd()
if not (ROOT / "notebooks-figures").exists():
    ROOT = ROOT.parent
sys.path.append(str(ROOT / "notebooks-figures"))

from figure_style import (
    RED, BLUE, VIOLET, ORANGE, GRAY, LIGHT_GRAY, BACKGROUND,
    DIRAC_MARKER_SIZE, MASS_MARKER_MIN_FACTOR, MASS_MARKER_MAX_FACTOR,
    TRANSPORT_LINE_MIN_WIDTH, POINT_EDGE_WIDTH,
    setup_matplotlib, figure_dir, save_pdf, remove_axes, padded_limits,
    interp_color,
)

setup_matplotlib()

NAME = "w1-graph-transport-flow"
OUT = figure_dir(NAME)


## A larger Delaunay graph and localized signed masses

We build a jittered $6\times6$ point set, orient each Delaunay edge by increasing vertex index, and solve the finite-dimensional Beckmann linear program with positive and negative parts of the signed edge flow.

In [ ]:

rng = np.random.default_rng(715)
xs = np.linspace(-1.75, 1.75, 6)
ys = np.linspace(-1.25, 1.25, 6)
points = np.array([[x, y] for y in ys for x in xs], dtype=float)
points += rng.normal(scale=[0.075, 0.070], size=points.shape)
n = len(points)

tri = Delaunay(points)
edges = set()
for simplex in tri.simplices:
    for a in range(3):
        for b in range(a + 1, 3):
            edges.add(tuple(sorted((int(simplex[a]), int(simplex[b])))))
edges = sorted(edges)
lengths = np.array([np.linalg.norm(points[i] - points[j]) for i, j in edges])

source_center = np.array([-1.22, 0.02])
target_center = np.array([1.24, -0.02])
source_idx = np.argsort(np.linalg.norm(points - source_center, axis=1))[:5]
target_idx = np.argsort(np.linalg.norm(points - target_center, axis=1))[:5]
source_weights = np.exp(-2.0 * np.linalg.norm(points[source_idx] - source_center, axis=1) ** 2)
target_weights = np.exp(-2.1 * np.linalg.norm(points[target_idx] - target_center, axis=1) ** 2)
source_weights = source_weights / source_weights.sum()
target_weights = target_weights / target_weights.sum()

alpha = np.zeros(n)
beta = np.zeros(n)
alpha[source_idx] = source_weights
beta[target_idx] = target_weights
r = alpha - beta

incidence = np.zeros((n, len(edges)))
for e, (i, j) in enumerate(edges):
    incidence[i, e] = 1.0
    incidence[j, e] = -1.0

objective = np.r_[lengths, lengths]
constraints = np.c_[incidence, -incidence]
res = linprog(
    objective,
    A_eq=constraints[:-1],
    b_eq=r[:-1],
    bounds=[(0.0, None)] * (2 * len(edges)),
    method="highs",
)
if not res.success:
    raise RuntimeError(res.message)
flow = res.x[: len(edges)] - res.x[len(edges) :]
assert np.linalg.norm(incidence @ flow - r, ord=np.inf) < 1e-9
active_edges = np.flatnonzero(np.abs(flow) > 1e-9)
flow_cost = float(np.dot(lengths, np.abs(flow)))
flow_cost, n, len(edges), len(active_edges)


## Exported panels

The first panel shows signed masses on the graph.  The second panel overlays the optimal edge flow.  Only circular markers are used for masses and vertices; edge widths encode $\sqrt{|m_e|}$.

In [ ]:

xlim_graph, ylim_graph = padded_limits(points, pad=0.10)


def draw_base_graph(ax, *, alpha_edges=0.48, lw=0.42):
    segments = [[points[i], points[j]] for i, j in edges]
    rgba = (*to_rgb(LIGHT_GRAY), alpha_edges)
    ax.add_collection(LineCollection(segments, colors=[rgba], linewidths=lw, zorder=0))
    ax.scatter(points[:, 0], points[:, 1], s=DIRAC_MARKER_SIZE * 0.22, marker="o", color=LIGHT_GRAY, edgecolor="none", linewidth=0, zorder=1)


def mass_sizes(weights):
    sizes = np.zeros_like(weights)
    mask = weights > 0
    if np.any(mask):
        sizes[mask] = DIRAC_MARKER_SIZE * (0.50 + 1.20 * weights[mask] / weights[mask].max())
    return sizes


def draw_masses(ax, *, alpha_scale=1.0):
    sa = mass_sizes(alpha) * alpha_scale
    sb = mass_sizes(beta) * alpha_scale
    ma = alpha > 0
    mb = beta > 0
    ax.scatter(points[ma, 0], points[ma, 1], s=sa[ma], marker="o", color=RED, edgecolor="none", linewidth=0, zorder=4)
    ax.scatter(points[mb, 0], points[mb, 1], s=sb[mb], marker="o", color=BLUE, edgecolor="none", linewidth=0, zorder=4)


def finish_graph(ax):
    ax.set_xlim(*xlim_graph)
    ax.set_ylim(*ylim_graph)
    ax.set_aspect("equal")
    remove_axes(ax)

fig, ax = plt.subplots(figsize=(2.28, 2.08))
draw_base_graph(ax, alpha_edges=0.62, lw=0.48)
draw_masses(ax)
finish_graph(ax)
save_pdf(fig, OUT / "masses.pdf", pad_inches=0.052)
plt.close(fig)

fig, ax = plt.subplots(figsize=(2.28, 2.08))
draw_base_graph(ax, alpha_edges=0.34, lw=0.38)
max_flow = max(float(np.max(np.abs(flow))), 1e-15)
for e in active_edges:
    i, j = edges[e]
    val = flow[e]
    start, end = (i, j) if val > 0 else (j, i)
    p0 = points[start].copy()
    p1 = points[end].copy()
    d = p1 - p0
    ell = np.linalg.norm(d)
    if ell == 0:
        continue
    u = d / ell
    p0 = p0 + 0.055 * u
    p1 = p1 - 0.055 * u
    normalized = abs(val) / max_flow
    width = TRANSPORT_LINE_MIN_WIDTH + (1.62 - TRANSPORT_LINE_MIN_WIDTH) * np.sqrt(normalized)
    arrow = FancyArrowPatch(
        p0,
        p1,
        arrowstyle="-|>",
        mutation_scale=5.4 + 2.0 * np.sqrt(normalized),
        linewidth=width,
        color=VIOLET,
        alpha=min(0.28 + 0.58 * normalized, 0.86),
        shrinkA=0,
        shrinkB=0,
        zorder=3,
    )
    ax.add_patch(arrow)

draw_masses(ax, alpha_scale=0.86)
finish_graph(ax)
save_pdf(fig, OUT / "flow.pdf", pad_inches=0.052)
plt.close(fig)
